In [ ]:
!pip install mediapipe==0.10.21 opencv-python

In [1]:
import os
import torch
import tarfile
import cv2
import numpy as np
import pandas as pd
#import mediapipe as mp
import matplotlib.pyplot as plt

In [ ]:
import mediapipe as mp

In [ ]:
os.listdir("/kaggle/input/datasets/toxicmender/20bn-jester")

**MediaPipe**

Initializes the MediaPipe Hands module and defines how each gesture will be represented for model training. Each folder contains 37 frames, and each frame contains 21 hand landmarks with (x, y, z) coordinates, resulting in 63 features per frame.

In [ ]:
from tqdm import tqdm

mp_hands = mp.solutions.hands

SEQUENCE_LENGTH = 37
NUM_LANDMARKS = 21
FEATURES_PER_FRAME = NUM_LANDMARKS * 3  # x, y, z = 63

**Extracting hand landmark features from a single image**

Function reads image, converts it to the correct color format (RGB), and processes it through the MediaPipe. If a hand is detected, it extracts all 21 landmark points and their (x, y, z) coordinates, resulting in a 63-dimensional feature vector. If no hand is detected or the image cannot be loaded, it returns a vector of zeros to maintain consistent input shape for the model.

In [ ]:
def extract_landmarks_from_image(image_path, hands):
    image = cv2.imread(image_path)

    if image is None:
        return np.zeros(FEATURES_PER_FRAME)

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = hands.process(image_rgb)

    if results.multi_hand_landmarks:
        hand_landmarks = results.multi_hand_landmarks[0]

        landmarks = []
        for lm in hand_landmarks.landmark:
            landmarks.extend([lm.x, lm.y, lm.z])

        return np.array(landmarks)

    else:
        return np.zeros(FEATURES_PER_FRAME)

**Processes a folder of image frames** (representing a single gesture video) and converts it into a fixed-length sequence of hand landmark features. It reads and sorts the frame images and applies the landmark extraction function to each frame. The resulting landmark vectors are combined into a sequence of shape (37, 63). If the folder contains fewer than 37 frames, the sequence is padded with zero vectors to maintain a consistent input size for model training.

In [ ]:
def process_frame_folder(folder_path, hands, sequence_length=SEQUENCE_LENGTH):
    frame_files = sorted(os.listdir(folder_path))

    # Keep only image files
    frame_files = [
        f for f in frame_files
        if f.lower().endswith(('.jpg'))
    ]

    # Take first 37 frames
    frame_files = frame_files[:sequence_length]

    
    sequence = []

    for frame in frame_files:
        frame_path = os.path.join(folder_path, frame)
        landmarks = extract_landmarks_from_image(frame_path, hands)
        sequence.append(landmarks)

    # If less than 37 frames, pad with zeros
    while len(sequence) < sequence_length:
        sequence.append(np.zeros(FEATURES_PER_FRAME))

    return np.array(sequence)

**Selects first 10000 folders for processing.**

In [ ]:
DATASET_PATH = "/kaggle/input/datasets/toxicmender/20bn-jester/Train"

all_folders = sorted([
    f for f in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, f))
])

subset_folders = all_folders[8000:10000]

In [ ]:
print(len(all_folders))      # should be ~50,000
print(len(subset_folders))   
print(subset_folders[:10])    # preview

**Load CSV file to map each video folder to its corresponding gesture class**

In [ ]:
csv_path = "/kaggle/input/datasets/toxicmender/20bn-jester/Train.csv"

df = pd.read_csv(csv_path)
label_dict = dict(zip(df['video_id'].astype(str), df['label_id']))
df

**Build training datasets**

Iterate through every folder and form sequence representation of hand gesture. The associated label for each video is retrieved using the label mapping, and both the feature sequence and label are stored in lists. These lists will later be converted into arrays for model training.

**Does not need to be rerun if the processed data has already been saved as .npy files.**

In [ ]:
'''

X = []
y = []

chunk_size = 1000
chunk_index = 1

with mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
) as hands:

    for i, folder_name in enumerate(tqdm(subset_folders)):

        folder_path = os.path.join(DATASET_PATH, folder_name)

        sequence = process_frame_folder(folder_path, hands)

        label_index = label_dict[folder_name]

        X.append(sequence)
        y.append(label_index)

        # Save every 1000 samples
        if (i + 1) % chunk_size == 0:

            X_chunk = np.array(X)
            y_chunk = np.array(y)

            np.save(f"X_chunk_{chunk_index}.npy", X_chunk)
            np.save(f"y_chunk_{chunk_index}.npy", y_chunk)

            print(f"Saved chunk {chunk_index}")

            # Clear memory
            X = []
            y = []

            chunk_index += 1

'''

**Converts collected feature sequences and labels into arrays for efficient processing**

In [ ]:
'''
X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)
'''

**Saves the processed feature data (X) and labels (y) as NumPy files**

In [ ]:
'''
np.save("X_jester_1000.npy", X)
np.save("y_jester_1000.npy", y)

print(X.shape)
print(y.shape)
'''

**Load saved data**

In [ ]:
'''
X = np.load("/kaggle/input/datasets/nghinguyenn/jester-1000/X_jester_1000.npy")
y = np.load("/kaggle/input/datasets/nghinguyenn/jester-1000/y_jester_1000.npy")

print(X.shape)
print(y.shape)
'''
X_chunks = []
y_chunks = []

for i in range(9):
    X_chunks.append(np.load(f"/kaggle/input/datasets/nghinguyenn/mediapipe-9000/X_chunk_{i}.npy"))
    y_chunks.append(np.load(f"/kaggle/input/datasets/nghinguyenn/mediapipe-9000/y_chunk_{i}.npy"))

X = np.concatenate(X_chunks, axis=0) # input (samples, frames, values)
y = np.concatenate(y_chunks, axis=0) # correct label (samples)

print(X.shape)
print(y.shape)
print(X[1])

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/toxicmender/20bn-jester/Validation.csv")
df

In [ ]:
result = df.set_index("label_id")["label"].to_dict()
result

**Train/validation split**

The training set of 800 data points is used to fit the model, while the validation set of 200 data points is used to evaluate its performance on unseen data during training. Stratification is applied to preserve the distribution of gesture classes across both sets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

**Defining Temporal Convolutional Network (TCN) model**

The model applies 1D convolutional layers across the time dimension to capture motion patterns between frames. Dilated convolutions are used to allow the network to learn both short-term and long-term dependencies in the gesture sequence. Batch normalization, ReLU activation, and dropout layers are included to improve training stability and reduce overfitting. The final layers aggregate the temporal information and output class probabilities for gesture classification.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv1D, BatchNormalization, ReLU, Dropout, GlobalAveragePooling1D, Dense

num_classes = len(np.unique(y))

model = Sequential([
    Input(shape=(37, 63)),

    Conv1D(64, kernel_size=3, padding='causal'), # Look at 3 consecutive frames at a time
    BatchNormalization(),
    ReLU(),
    Dropout(0.3),

    Conv1D(64, kernel_size=3, padding='causal', dilation_rate=2),
    BatchNormalization(),
    ReLU(),
    Dropout(0.3),

    Conv1D(128, kernel_size=3, padding='causal', dilation_rate=4), # lets the model look farther apart in time (frame 1,5,9)
    BatchNormalization(),
    ReLU(),
    Dropout(0.3),

    GlobalAveragePooling1D(),

    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

**Compiles the TCN model**

Specifying the optimizer, loss function, and evaluation metric. The Adam optimizer is used for efficient training, and sparse categorical crossentropy is applied since the labels are integer-encoded. Accuracy is tracked to monitor model performance during training. The model summary provide an overview of the network architecture, including layer types, output shapes, and the number of trainable parameters.

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

**Trains the TCN model**

Iterating over the data for multiple epochs (full pass through entire training dataset). Validation data is used during training to evaluate performance on unseen samples and monitor generalization. The training process outputs metrics such as loss and accuracy for both training and validation sets at each epoch.

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs = 20,    # model sees all 1000 samples 20 times
    batch_size = 32 # 32 samples at a time before updating itself.
)

**Training results - 1000 data points:**
- Training accuracy increased from 6% to 48%
- Best epoch: 17 - val_accuracy = 0.4100

**Training results - 9000 data points:**
- Training accuracy increased from 6% to 68%
- Best epoch: 19 - val_accuracy = 0.7033

**Save model**

In [ ]:
model.save("tcn_gesture_model.keras")

In [ ]:
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('TCN Training vs Validation Accuracy')
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('TCN Training vs Validation Loss')
plt.legend()
plt.show()

**Testing model**

In [ ]:
# Create label_id → label name mapping
id_to_label = dict(zip(df['label_id'], df['label']))

test_folder = "/kaggle/input/datasets/toxicmender/20bn-jester/Train/100620" #change number

with mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.5
) as hands:
    test_sequence = process_frame_folder(test_folder, hands)

test_sequence = np.expand_dims(test_sequence, axis=0)

print(test_sequence.shape)

**Predict label ID and gesture**

In [ ]:
pred_probs = model.predict(test_sequence)

pred_label_id = np.argmax(pred_probs)
pred_label_name = id_to_label[pred_label_id]

print("Predicted label ID:", pred_label_id)
print("Predicted gesture:", pred_label_name)
print("Confidence:", pred_probs[0][pred_label_id])

**True label ID and gesture**

In [ ]:
folder_name = os.path.basename(test_folder)

true_label_id = label_dict[folder_name]
true_label_name = id_to_label[true_label_id]

print("True label ID:", true_label_id)
print("True gesture:", true_label_name)

In [ ]:
np.save("X_jester_9000.npy", X)
np.save("y_jester_9000.npy", y)